# Build LangGraph Agents with Kagenti ADK

This notebook walks through building each agent using LangGraph for logic and Kagenti ADK for A2A compliance.
These agents integrate with the AGENTS.md harness as execution components.

## 1. Load Environment

In [1]:
import os
import sys
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

# Add agents directory to path for imports
agents_dir = os.path.join(os.path.dirname(os.getcwd()), "agents")
print(f"Agents directory: {agents_dir}")
print("✅ Environment loaded")

Agents directory: /Users/hyochoi/dev/rhoai-cusom-research-lab/agents
✅ Environment loaded


## 2. Understand the Agent Pattern

Every agent in this lab follows the same structure:

```python
from kagenti_adk.server import Server
from kagenti_adk.a2a.types import AgentMessage
from langgraph.graph import StateGraph, END

# 1. Define state schema
class AgentState(TypedDict):
    input: str
    output: str

# 2. Build LangGraph
graph = StateGraph(AgentState)
graph.add_node("process", process_fn)
graph.set_entry_point("process")
graph.add_edge("process", END)
compiled = graph.compile()

# 3. Wrap with Kagenti ADK
server = Server()

@server.agent()
async def my_agent(input: Message, context: RunContext):
    result = await compiled.ainvoke({...})
    yield AgentMessage(text=result["output"])
```

## 3. Verify Agent Code Structure

Check that all agent files are properly structured.

In [2]:
expected_agents = ["orchestrator", "doc_processor", "researcher", "writer", "reviewer"]
expected_files = ["agent.py", "tools.py", "Dockerfile"]

for agent in expected_agents:
    agent_path = os.path.join(agents_dir, agent)
    if os.path.isdir(agent_path):
        files = os.listdir(agent_path)
        missing = [f for f in expected_files if f not in files]
        if missing:
            print(f"  ⚠️ {agent}: missing {missing}")
        else:
            print(f"  ✅ {agent}: all files present")
    else:
        print(f"  ❌ {agent}: directory not found")

print("\n✅ Agent structure verification complete")

  ✅ orchestrator: all files present
  ✅ doc_processor: all files present
  ✅ researcher: all files present
  ✅ writer: all files present
  ✅ reviewer: all files present

✅ Agent structure verification complete


## 4. Test Document Processor Agent Logic

Test the LangGraph workflow for the Document Processor without running the A2A server.

In [3]:
sys.path.insert(0, os.path.join(agents_dir, "doc_processor"))

from agents.doc_processor.tools import get_document_status

# Test status check (should return "not found" for non-existent doc)
status = get_document_status("nonexistent-id")
print(f"Status check for nonexistent doc: {status}")
print("✅ Document Processor tools accessible")

Status check for nonexistent doc: not found
✅ Document Processor tools accessible


## 5. Test Researcher Agent Logic

Test the RAG search functionality.

In [4]:
sys.path.insert(0, os.path.join(agents_dir, "researcher"))

from agents.researcher.tools import semantic_search, rewrite_query

# Test query rewriting
test_query = "How does the document processing pipeline work?"
rewritten = rewrite_query(test_query)
print(f"Original: {test_query}")
print(f"Rewritten queries: {rewritten}")

# Test semantic search
results = semantic_search(test_query, top_k=3)
print(f"\nSearch results: {len(results)} passages found")
for r in results:
    print(f"  - {r['document_name']} (chunk {r['chunk_index']}, sim={r['similarity']:.3f})")

print("\n✅ Researcher tools working")

Original: How does the document processing pipeline work?
Rewritten queries: ['How does the document processing pipeline work?']

Search results: 3 passages found
  - sample.pdf (chunk 4, sim=0.713)
  - sample.pdf (chunk 10, sim=0.681)
  - sample.pdf (chunk 11, sim=0.651)

✅ Researcher tools working


## 6. Test Writer Agent Logic

In [5]:
sys.path.insert(0, os.path.join(agents_dir, "writer"))

from agents.writer.tools import plan_report_structure

context = "\n".join([r["content"][:200] for r in results]) if results else "No context available."
plan = plan_report_structure(test_query, context)
print("Report structure plan:")
print(plan)
print("\n✅ Writer tools working")

APIConnectionError: Connection error.

## 7. Test Reviewer Agent Logic

In [ ]:
sys.path.insert(0, os.path.join(agents_dir, "reviewer"))

from agents.reviewer.tools import score_quality

sample_report = """# Document Processing Pipeline Analysis

## Executive Summary
The document processing pipeline uses Docling for parsing and pgvector for storage [Source 1].

## Key Findings
1. Docling supports multiple formats including PDF, DOCX, and PPTX.
2. Semantic chunking preserves document structure.
3. pgvector enables efficient similarity search.

## Conclusion
The pipeline provides robust document processing capabilities.
"""

score = score_quality(sample_report)
print(f"Quality score: {score}/10")
print("✅ Reviewer tools working")